In [4]:
import pandas as pd;
df = pd.read_csv('kent_personal_care_locations.csv');
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 579 entries, 0 to 578
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   locationId           579 non-null    object 
 1   providerId           579 non-null    object 
 2   locationName         579 non-null    object 
 3   registrationStatus   579 non-null    object 
 4   registrationDate     579 non-null    object 
 5   addressLine1         579 non-null    object 
 6   town                 579 non-null    object 
 7   postcode             579 non-null    object 
 8   region               579 non-null    object 
 9   localAuthority       579 non-null    object 
 10  constituency         579 non-null    object 
 11  latitude             579 non-null    float64
 12  longitude            579 non-null    float64
 13  phoneNumber          519 non-null    float64
 14  regulatedActivities  579 non-null    object 
 15  gacServiceTypes      579 non-null    obj

In [2]:
print(f"Starting rows: {len(df)}")

# 1. Keep only currently registered locations (drop deregistered/inactive)
df = df[df['registrationStatus'] == 'Registered'].copy()
print(f"After dropping non-Registered: {len(df)}")

# 2. Drop exact duplicate locationIds, just in case
before = len(df)
df = df.drop_duplicates(subset='locationId')
print(f"After deduplicating locationId: {len(df)} (removed {before - len(df)})")

# 3. Standardise postcodes: uppercase, single space before the last 3 chars
def clean_postcode(pc):
    if not isinstance(pc, str) or not pc.strip():
        return None
    pc = pc.strip().upper().replace(" ", "")
    if len(pc) < 5:
        return pc  # too short to safely reformat, leave as-is
    return f"{pc[:-3]} {pc[-3:]}"

df['postcode'] = df['postcode'].apply(clean_postcode)
print(f"Rows with missing/unparseable postcode: {df['postcode'].isna().sum()}")

# 4. Standardise text fields: trim whitespace
text_cols = ['locationName', 'town', 'localAuthority', 'region', 'constituency']
for col in text_cols:
    df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)

# 5. Handle nulls in key fields - flag rather than silently drop
key_fields = ['locationId', 'locationName', 'postcode', 'overallRating']
null_report = df[key_fields].isna().sum()
print("\nNulls in key fields:")
print(null_report)

# 6. Standardise overallRating values
if 'overallRating' in df.columns:
    df['overallRating'] = df['overallRating'].apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )
    print("\nRating value counts:")
    print(df['overallRating'].value_counts(dropna=False))

# 7. Parse dates properly
df['registrationDate'] = pd.to_datetime(df['registrationDate'], errors='coerce')
df['overallRatingDate'] = pd.to_datetime(df['overallRatingDate'], errors='coerce')

print(f"\nFinal cleaned row count: {len(df)}")
df.head()

Starting rows: 579
After dropping non-Registered: 579
After deduplicating locationId: 579 (removed 0)
Rows with missing/unparseable postcode: 0

Nulls in key fields:
locationId         0
locationName       0
postcode           0
overallRating    341
dtype: int64

Rating value counts:
overallRating
NaN                     341
Good                    186
Requires improvement     42
Outstanding              10
Name: count, dtype: int64

Final cleaned row count: 579


/tmp/ipykernel_375/3576593014.py:44: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['registrationDate'] = pd.to_datetime(df['registrationDate'], errors='coerce')
/tmp/ipykernel_375/3576593014.py:45: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['overallRatingDate'] = pd.to_datetime(df['overallRatingDate'], errors='coerce')


,locationId,providerId,locationName,registrationStatus,registrationDate,addressLine1,town,postcode,region,localAuthority,constituency,latitude,longitude,phoneNumber,regulatedActivities,gacServiceTypes,overallRating,overallRatingDate,numberOfBeds,website
0,1-10064258273,1-9491037344,Kent Case Management Ltd,Registered,2020-12-30,"The Stables, Bradbourne House,",West Malling,ME19 6DZ,South East,Kent,Maidstone and Malling,51.295547,0.441755,7.741498e+09,Personal care,Homecare agencies,Good,2022-03-18,0.0,www.kentcasemanagement.co.uk
1,1-10086956930,1-8974776775,Assured Healthcare Limited,Registered,2020-12-24,75 Wiltshire Close,Chatham,ME5 7SS,South East,Medway,Chatham and Aylesford,51.365265,0.547372,1.634320e+09,"Personal care; Treatment of disease, disorder ...",Community services - Healthcare; Homecare agen...,NaN,NaT,0.0,www.assuredhc.co.uk
2,1-10087632956,1-6192052130,Royalcare- Thanet,Registered,2020-12-23,Kent Innovation Centre,Broadstairs,CT10 2QQ,South East,Kent,East Thanet,51.358932,1.406302,1.843838e+09,Personal care,Homecare agencies,Requires improvement,2022-05-25,0.0,www.royalcare24.co.uk
3,1-10204301199,1-118164017,Priory Supported Living Kent,Registered,2021-02-25,Buckland,Dover,CT17 0TQ,South East,Kent,Dover and Deal,51.135385,1.298625,1.304202e+09,Personal care,Homecare agencies; Supported living,Good,2022-11-11,0.0,www.prioryadultcare.co.uk
4,1-10200463066,1-3982604534,PCAS Kent Ltd,Registered,2021-01-14,Unit 5,Faversham,ME13 8GD,South East,Kent,Faversham and Mid Kent,51.310679,0.898310,3.300536e+09,Personal care,Homecare agencies,NaN,NaT,0.0,NaN


In [5]:
print(df.isna().sum())

locationId               0
providerId               0
locationName             0
registrationStatus       0
registrationDate         0
addressLine1             0
town                     0
postcode                 0
region                   0
localAuthority           0
constituency             0
latitude                 0
longitude                0
phoneNumber             60
regulatedActivities      0
gacServiceTypes          0
overallRating          341
overallRatingDate      341
numberOfBeds            13
website                254
dtype: int64
